# Statistical Significance Tests for Activation Steering Paper

**Muc tieu:** Tinh toan p-value, confidence interval, effect size, va Repetition-Penalized ROUGE-L
tu du lieu per-sample da co (Phase 3B 80tok + Phase 3C 200tok).

**Input can thiet (Add vao Kaggle Input):**
1. Output cua notebook `kaggle-phase3b1-control-methods`
2. Output cua notebook `kaggle-phase3b2-control-methods`
3. Output cua notebook `kaggle-phase3c1-main-methods-200token`
4. Output cua notebook `kaggle-phase3c2-control-methods-200tok`

**Thoi gian chay:** ~5-10 phut (chi tinh toan thong ke, khong load model AI)

---

In [ ]:
# Cell 1: Install Dependencies
!pip install -q rouge-score bert-score scipy numpy tqdm
print('Dependencies installed!')

In [ ]:
# Cell 2: Imports & Config
import os, json, glob, random, re
import numpy as np
from scipy import stats
from collections import defaultdict
from rouge_score import rouge_scorer

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
OUTPUT_DIR = '/kaggle/working'

N_BOOTSTRAP = 10000  # Number of bootstrap resamples
ALPHA_CI = 0.05     # 95% confidence interval

print(f'Config: N_BOOTSTRAP={N_BOOTSTRAP}, CI={1-ALPHA_CI:.0%}')

In [ ]:
# Cell 3: Load all per-sample data from Phase 3B and 3C outputs
print('='*70)
print('LOADING PER-SAMPLE DATA FROM ALL PHASE OUTPUTS')
print('='*70)

def find_json_files(pattern):
    """Search for JSON files matching pattern in /kaggle/input/"""
    paths = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    return paths

# Search for all generated text files
all_json_files = glob.glob('/kaggle/input/**/*.json', recursive=True)
print(f'\nFound {len(all_json_files)} JSON files in /kaggle/input/:')
for f in sorted(all_json_files):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {f} ({size_kb:.1f} KB)')

# Load generated texts (per-sample data)
data_80tok = {}   # {method_name: [{'idx':..., 'generated':..., 'reference':..., 'rouge_l':...}, ...]}
data_200tok = {}

# Phase 3B: 80 tokens
for pattern in ['*phase3b1*generated*', '*phase3b1*texts*', '*phase3b1*main*']:
    paths = find_json_files(f'{pattern}.json')
    for p in paths:
        try:
            d = json.load(open(p, 'r', encoding='utf-8'))
            if isinstance(d, dict):
                for key, records in d.items():
                    if isinstance(records, list) and len(records) > 10 and 'generated' in records[0]:
                        data_80tok[key] = records
                        print(f'  Loaded 80tok [{key}]: {len(records)} samples from {os.path.basename(p)}')
        except: pass

for pattern in ['*phase3b2*generated*', '*phase3b2*texts*', '*phase3b2*control*']:
    paths = find_json_files(f'{pattern}.json')
    for p in paths:
        try:
            d = json.load(open(p, 'r', encoding='utf-8'))
            if isinstance(d, dict):
                for key, records in d.items():
                    if isinstance(records, list) and len(records) > 10 and 'generated' in records[0]:
                        data_80tok[key] = records
                        print(f'  Loaded 80tok [{key}]: {len(records)} samples from {os.path.basename(p)}')
        except: pass

# Phase 3C: 200 tokens
for pattern in ['*phase3c1*generated*', '*phase3c1*texts*']:
    paths = find_json_files(f'{pattern}.json')
    for p in paths:
        try:
            d = json.load(open(p, 'r', encoding='utf-8'))
            if isinstance(d, dict):
                for key, records in d.items():
                    if isinstance(records, list) and len(records) > 10 and 'generated' in records[0]:
                        data_200tok[key] = records
                        print(f'  Loaded 200tok [{key}]: {len(records)} samples from {os.path.basename(p)}')
        except: pass

for pattern in ['*phase3c2*generated*', '*phase3c2*texts*']:
    paths = find_json_files(f'{pattern}.json')
    for p in paths:
        try:
            d = json.load(open(p, 'r', encoding='utf-8'))
            if isinstance(d, dict):
                for key, records in d.items():
                    if isinstance(records, list) and len(records) > 10 and 'generated' in records[0]:
                        data_200tok[key] = records
                        print(f'  Loaded 200tok [{key}]: {len(records)} samples from {os.path.basename(p)}')
        except: pass

# Fallback: try loading ANY large JSON with per-sample data
if not data_80tok and not data_200tok:
    print('\n[FALLBACK] Trying to load from all available JSON files...')
    for f in all_json_files:
        try:
            d = json.load(open(f, 'r', encoding='utf-8'))
            if isinstance(d, dict):
                for key, records in d.items():
                    if isinstance(records, list) and len(records) > 10 and isinstance(records[0], dict) and 'generated' in records[0]:
                        tok_type = '200tok' if '200' in f or '200' in key else '80tok'
                        target = data_200tok if tok_type == '200tok' else data_80tok
                        target[key] = records
                        print(f'  Loaded {tok_type} [{key}]: {len(records)} samples from {os.path.basename(f)}')
        except: pass

print(f'\n--- Summary ---')
print(f'80-token methods loaded: {list(data_80tok.keys())}')
print(f'200-token methods loaded: {list(data_200tok.keys())}')

In [ ]:
# Cell 4: Recompute per-sample metrics (ROUGE-L, Rep4gram, Dedup-ROUGE-L)
print('='*70)
print('RECOMPUTING PER-SAMPLE METRICS')
print('='*70)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_rep4gram(text):
    """Compute 4-gram repetition ratio"""
    words = text.split()
    if len(words) < 4: return 0.0
    ngrams = [tuple(words[i:i+4]) for i in range(len(words)-3)]
    return 1.0 - len(set(ngrams))/len(ngrams) if ngrams else 0.0

def deduplicate_text(text):
    """Remove consecutive repeated phrases from text"""
    sentences = re.split(r'[.!?;]\s*', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    if not sentences: return text
    deduped = [sentences[0]]
    for s in sentences[1:]:
        if s != deduped[-1]:
            deduped.append(s)
    result = '. '.join(deduped)
    # Also remove word-level consecutive repetition
    words = result.split()
    if len(words) < 8: return result
    clean_words = list(words[:4])
    for i in range(4, len(words)):
        chunk = tuple(words[i-3:i+1])
        prev_chunk = tuple(words[i-7:i-3]) if i >= 7 else None
        if chunk != prev_chunk:
            clean_words.append(words[i])
    return ' '.join(clean_words)

def enrich_metrics(data_dict, label=''):
    """Recompute and add per-sample metrics"""
    for method, records in data_dict.items():
        for r in records:
            gen = r.get('generated', '')
            ref = r.get('reference', '')
            # Original ROUGE-L (verify or recompute)
            rl = scorer.score(ref, gen)['rougeL'].fmeasure * 100
            r['rouge_l_recomputed'] = rl
            # 4-gram repetition
            r['rep4gram'] = compute_rep4gram(gen)
            # Deduplicated ROUGE-L
            gen_dedup = deduplicate_text(gen)
            r['rouge_l_dedup'] = scorer.score(ref, gen_dedup)['rougeL'].fmeasure * 100
            # Repetition-Penalized ROUGE-L
            r['rouge_l_penalized'] = rl * (1.0 - r['rep4gram'])
        avg_rl = np.mean([r['rouge_l_recomputed'] for r in records])
        avg_rep = np.mean([r['rep4gram'] for r in records])
        avg_dedup = np.mean([r['rouge_l_dedup'] for r in records])
        avg_pen = np.mean([r['rouge_l_penalized'] for r in records])
        print(f'  [{label}] {method:<30} ROUGE-L={avg_rl:.2f}%  Rep4={avg_rep:.4f}  Dedup-RL={avg_dedup:.2f}%  Pen-RL={avg_pen:.2f}%')

if data_80tok:
    print('\n--- 80-token metrics ---')
    enrich_metrics(data_80tok, '80tok')
if data_200tok:
    print('\n--- 200-token metrics ---')
    enrich_metrics(data_200tok, '200tok')

In [ ]:
# Cell 5: Statistical Test Functions
print('='*70)
print('STATISTICAL TEST FUNCTIONS')
print('='*70)

def paired_bootstrap_test(scores_a, scores_b, n_bootstrap=10000, seed=42):
    """
    Paired Bootstrap Test (standard in NLP evaluation).
    Tests H0: mean(scores_a) <= mean(scores_b)
    Returns p-value and 95% CI for the difference (a - b).
    """
    rng = np.random.RandomState(seed)
    n = len(scores_a)
    assert len(scores_b) == n, f'Mismatched lengths: {len(scores_a)} vs {len(scores_b)}'
    
    observed_diff = np.mean(scores_a) - np.mean(scores_b)
    
    # Bootstrap resampling
    boot_diffs = []
    for _ in range(n_bootstrap):
        indices = rng.randint(0, n, size=n)
        boot_a = np.array(scores_a)[indices]
        boot_b = np.array(scores_b)[indices]
        boot_diffs.append(np.mean(boot_a) - np.mean(boot_b))
    boot_diffs = np.array(boot_diffs)
    
    # Two-sided p-value
    p_value = np.mean(boot_diffs <= 0) if observed_diff > 0 else np.mean(boot_diffs >= 0)
    
    # 95% CI
    ci_lower = np.percentile(boot_diffs, 2.5)
    ci_upper = np.percentile(boot_diffs, 97.5)
    
    return {
        'observed_diff': observed_diff,
        'p_value': p_value,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'significant': p_value < 0.05
    }

def wilcoxon_test(scores_a, scores_b):
    """
    Wilcoxon Signed-Rank Test (non-parametric paired test).
    """
    diffs = np.array(scores_a) - np.array(scores_b)
    # Remove zeros (ties)
    nonzero_diffs = diffs[diffs != 0]
    if len(nonzero_diffs) < 10:
        return {'statistic': 0, 'p_value': 1.0, 'significant': False}
    stat, p_value = stats.wilcoxon(nonzero_diffs, alternative='two-sided')
    return {
        'statistic': stat,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

def cohens_d(scores_a, scores_b):
    """
    Cohen's d effect size for paired samples.
    """
    diffs = np.array(scores_a) - np.array(scores_b)
    d = np.mean(diffs) / np.std(diffs, ddof=1) if np.std(diffs, ddof=1) > 0 else 0
    # Interpretation
    abs_d = abs(d)
    if abs_d < 0.2: interp = 'negligible'
    elif abs_d < 0.5: interp = 'small'
    elif abs_d < 0.8: interp = 'medium'
    else: interp = 'large'
    return {'d': d, 'interpretation': interp}

def run_all_tests(scores_a, scores_b, name_a, name_b, metric_name='ROUGE-L'):
    """Run all statistical tests between two sets of paired scores."""
    boot = paired_bootstrap_test(scores_a, scores_b, N_BOOTSTRAP)
    wilc = wilcoxon_test(scores_a, scores_b)
    cd = cohens_d(scores_a, scores_b)
    
    mean_a = np.mean(scores_a)
    mean_b = np.mean(scores_b)
    std_a = np.std(scores_a, ddof=1)
    std_b = np.std(scores_b, ddof=1)
    
    sig_symbol = '***' if boot['p_value'] < 0.001 else ('**' if boot['p_value'] < 0.01 else ('*' if boot['p_value'] < 0.05 else 'n.s.'))
    
    print(f'\n  {name_a} vs {name_b} [{metric_name}]')
    print(f'    Mean A: {mean_a:.4f} (+/- {std_a:.4f})  |  Mean B: {mean_b:.4f} (+/- {std_b:.4f})')
    print(f'    Diff (A-B): {boot["observed_diff"]:+.4f}')
    print(f'    Bootstrap: p={boot["p_value"]:.6f} {sig_symbol}, 95% CI=[{boot["ci_lower"]:+.4f}, {boot["ci_upper"]:+.4f}]')
    print(f'    Wilcoxon:  p={wilc["p_value"]:.6f} {"*" if wilc["significant"] else "n.s."}')
    print(f'    Cohen\'s d: {cd["d"]:+.4f} ({cd["interpretation"]})')
    
    return {
        'comparison': f'{name_a} vs {name_b}',
        'metric': metric_name,
        'mean_a': mean_a, 'mean_b': mean_b,
        'diff': boot['observed_diff'],
        'bootstrap_p': boot['p_value'],
        'bootstrap_ci': (boot['ci_lower'], boot['ci_upper']),
        'wilcoxon_p': wilc['p_value'],
        'cohens_d': cd['d'],
        'effect_size': cd['interpretation'],
        'significant': boot['significant']
    }

print('Statistical test functions ready.')

In [ ]:
# Cell 6: STATISTICAL TESTS FOR 80-TOKEN RESULTS (Phase 3B)
print('='*80)
print('STATISTICAL SIGNIFICANCE TESTS: 80-TOKEN RESULTS (Phase 3B)')
print('='*80)

all_test_results_80 = []

if data_80tok:
    methods_80 = list(data_80tok.keys())
    print(f'Available 80tok methods: {methods_80}')
    
    # Identify method keys (flexible matching)
    def find_key(data, patterns):
        for p in patterns:
            for k in data.keys():
                if p in k.lower():
                    return k
        return None
    
    baseline_key = find_key(data_80tok, ['baseline', 'vanilla'])
    es_best_key = find_key(data_80tok, ['es_best', 'best_es', 'a18', 'alpha18'])
    es_runner_key = find_key(data_80tok, ['es_runner', 'runner_es', 'runner', 'a15', 'alpha15'])
    full_key = find_key(data_80tok, ['full_steer', 'full'])
    rand_key = find_key(data_80tok, ['random', 'rand', 'ctrl_rand'])
    sign_key = find_key(data_80tok, ['sign', 'flip', 'ctrl_sign'])
    
    print(f'\nIdentified keys:')
    print(f'  Baseline: {baseline_key}')
    print(f'  Best ES:  {es_best_key}')
    print(f'  Runner ES: {es_runner_key}')
    print(f'  Full Steer: {full_key}')
    print(f'  Ctrl Random: {rand_key}')
    print(f'  Ctrl SignFlip: {sign_key}')
    
    # Define comparisons
    comparisons_80 = []
    if es_runner_key and full_key:
        comparisons_80.append(('Early-Stop (a15,K16)', es_runner_key, 'Full Steering', full_key))
    if es_best_key and full_key:
        comparisons_80.append(('Early-Stop (a18,K16)', es_best_key, 'Full Steering', full_key))
    if es_runner_key and baseline_key:
        comparisons_80.append(('Early-Stop (a15,K16)', es_runner_key, 'Baseline', baseline_key))
    if es_best_key and baseline_key:
        comparisons_80.append(('Early-Stop (a18,K16)', es_best_key, 'Baseline', baseline_key))
    if full_key and baseline_key:
        comparisons_80.append(('Full Steering', full_key, 'Baseline', baseline_key))
    if baseline_key and rand_key:
        comparisons_80.append(('Baseline', baseline_key, 'Ctrl Random', rand_key))
    if baseline_key and sign_key:
        comparisons_80.append(('Baseline', baseline_key, 'Ctrl SignFlip', sign_key))
    
    # Run tests on ROUGE-L
    print('\n' + '-'*60)
    print('A. ROUGE-L Tests (80 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_80:
        scores_a = [r['rouge_l_recomputed'] for r in data_80tok[key_a]]
        scores_b = [r['rouge_l_recomputed'] for r in data_80tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, 'ROUGE-L')
        all_test_results_80.append(result)
    
    # Run tests on Repetition-Penalized ROUGE-L
    print('\n' + '-'*60)
    print('B. Repetition-Penalized ROUGE-L Tests (80 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_80:
        scores_a = [r['rouge_l_penalized'] for r in data_80tok[key_a]]
        scores_b = [r['rouge_l_penalized'] for r in data_80tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, 'Penalized-ROUGE-L')
        all_test_results_80.append(result)
    
    # Run tests on Dedup ROUGE-L
    print('\n' + '-'*60)
    print('C. Deduplicated ROUGE-L Tests (80 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_80:
        scores_a = [r['rouge_l_dedup'] for r in data_80tok[key_a]]
        scores_b = [r['rouge_l_dedup'] for r in data_80tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, 'Dedup-ROUGE-L')
        all_test_results_80.append(result)
    
    # Run tests on Rep4gram
    print('\n' + '-'*60)
    print('D. 4-gram Repetition Tests (80 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_80:
        scores_a = [r['rep4gram'] for r in data_80tok[key_a]]
        scores_b = [r['rep4gram'] for r in data_80tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, '4gram-Rep')
        all_test_results_80.append(result)
else:
    print('WARNING: No 80-token data found!')

In [ ]:
# Cell 7: STATISTICAL TESTS FOR 200-TOKEN RESULTS (Phase 3C)
print('='*80)
print('STATISTICAL SIGNIFICANCE TESTS: 200-TOKEN RESULTS (Phase 3C)')
print('='*80)

all_test_results_200 = []

if data_200tok:
    methods_200 = list(data_200tok.keys())
    print(f'Available 200tok methods: {methods_200}')
    
    baseline_key = find_key(data_200tok, ['baseline', 'vanilla'])
    es_best_key = find_key(data_200tok, ['es_best', 'best_es', 'a18', 'alpha18'])
    es_runner_key = find_key(data_200tok, ['es_runner', 'runner_es', 'runner', 'a15', 'alpha15'])
    full_key = find_key(data_200tok, ['full_steer', 'full'])
    rand_key = find_key(data_200tok, ['random', 'rand', 'ctrl_rand'])
    sign_key = find_key(data_200tok, ['sign', 'flip', 'ctrl_sign'])
    
    print(f'\nIdentified keys:')
    print(f'  Baseline: {baseline_key}')
    print(f'  Best ES:  {es_best_key}')
    print(f'  Runner ES: {es_runner_key}')
    print(f'  Full Steer: {full_key}')
    print(f'  Ctrl Random: {rand_key}')
    print(f'  Ctrl SignFlip: {sign_key}')
    
    comparisons_200 = []
    if es_best_key and full_key:
        comparisons_200.append(('Early-Stop (a18,K16)', es_best_key, 'Full Steering', full_key))
    if es_runner_key and full_key:
        comparisons_200.append(('Early-Stop (a15,K16)', es_runner_key, 'Full Steering', full_key))
    if es_best_key and baseline_key:
        comparisons_200.append(('Early-Stop (a18,K16)', es_best_key, 'Baseline', baseline_key))
    if full_key and baseline_key:
        comparisons_200.append(('Full Steering', full_key, 'Baseline', baseline_key))
    if baseline_key and rand_key:
        comparisons_200.append(('Baseline', baseline_key, 'Ctrl Random', rand_key))
    if baseline_key and sign_key:
        comparisons_200.append(('Baseline', baseline_key, 'Ctrl SignFlip', sign_key))
    
    # ROUGE-L
    print('\n' + '-'*60)
    print('A. ROUGE-L Tests (200 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_200:
        scores_a = [r['rouge_l_recomputed'] for r in data_200tok[key_a]]
        scores_b = [r['rouge_l_recomputed'] for r in data_200tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, 'ROUGE-L')
        all_test_results_200.append(result)
    
    # Penalized ROUGE-L
    print('\n' + '-'*60)
    print('B. Repetition-Penalized ROUGE-L Tests (200 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_200:
        scores_a = [r['rouge_l_penalized'] for r in data_200tok[key_a]]
        scores_b = [r['rouge_l_penalized'] for r in data_200tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, 'Penalized-ROUGE-L')
        all_test_results_200.append(result)
    
    # Dedup ROUGE-L
    print('\n' + '-'*60)
    print('C. Deduplicated ROUGE-L Tests (200 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_200:
        scores_a = [r['rouge_l_dedup'] for r in data_200tok[key_a]]
        scores_b = [r['rouge_l_dedup'] for r in data_200tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, 'Dedup-ROUGE-L')
        all_test_results_200.append(result)
    
    # Rep4gram
    print('\n' + '-'*60)
    print('D. 4-gram Repetition Tests (200 tokens)')
    print('-'*60)
    for name_a, key_a, name_b, key_b in comparisons_200:
        scores_a = [r['rep4gram'] for r in data_200tok[key_a]]
        scores_b = [r['rep4gram'] for r in data_200tok[key_b]]
        result = run_all_tests(scores_a, scores_b, name_a, name_b, '4gram-Rep')
        all_test_results_200.append(result)
else:
    print('WARNING: No 200-token data found!')

In [ ]:
# Cell 8: Compute BERTScore per-sample & run statistical tests
print('='*80)
print('BERTSCORE PER-SAMPLE COMPUTATION & STATISTICAL TESTS')
print('='*80)

try:
    from bert_score import score as bert_score_fn
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {device}')
    
    def compute_bertscore_for_data(data_dict, label=''):
        for method, records in data_dict.items():
            refs = [r['reference'] for r in records]
            hyps = [r['generated'] for r in records]
            print(f'  Computing BERTScore for [{label}] {method} ({len(records)} samples)...')
            P, R, F1 = bert_score_fn(hyps, refs, model_type='bert-base-multilingual-cased',
                                      num_layers=9, verbose=False, device=device)
            for r, bs in zip(records, F1.tolist()):
                r['bertscore_f1'] = bs
            print(f'    Mean BERTScore F1: {np.mean(F1.tolist()):.4f}')
    
    if data_80tok:
        print('\n--- Computing BERTScore for 80-token data ---')
        compute_bertscore_for_data(data_80tok, '80tok')
    if data_200tok:
        print('\n--- Computing BERTScore for 200-token data ---')
        compute_bertscore_for_data(data_200tok, '200tok')
    
    # Run BERTScore statistical tests
    all_bert_results_80 = []
    all_bert_results_200 = []
    
    if data_80tok:
        print('\n' + '-'*60)
        print('E. BERTScore F1 Tests (80 tokens)')
        print('-'*60)
        for name_a, key_a, name_b, key_b in comparisons_80:
            scores_a = [r['bertscore_f1'] for r in data_80tok[key_a]]
            scores_b = [r['bertscore_f1'] for r in data_80tok[key_b]]
            result = run_all_tests(scores_a, scores_b, name_a, name_b, 'BERTScore-F1')
            all_bert_results_80.append(result)
    
    if data_200tok:
        print('\n' + '-'*60)
        print('E. BERTScore F1 Tests (200 tokens)')
        print('-'*60)
        for name_a, key_a, name_b, key_b in comparisons_200:
            scores_a = [r['bertscore_f1'] for r in data_200tok[key_a]]
            scores_b = [r['bertscore_f1'] for r in data_200tok[key_b]]
            result = run_all_tests(scores_a, scores_b, name_a, name_b, 'BERTScore-F1')
            all_bert_results_200.append(result)
    
    print('\nBERTScore analysis completed!')
except ImportError:
    print('WARNING: bert-score not available. Skipping BERTScore tests.')
    all_bert_results_80 = []
    all_bert_results_200 = []

In [ ]:
# Cell 9: PUBLICATION-READY SUMMARY TABLE
print('='*100)
print('PUBLICATION-READY STATISTICAL SIGNIFICANCE TABLE')
print('='*100)

def print_summary_table(results, title):
    print(f'\n{title}')
    print('-'*100)
    header = f'{"Comparison":<45} {"Metric":<18} {"Diff":>8} {"Boot p":>10} {"Wilcoxon p":>10} {"Cohen d":>9} {"Sig":>5}'
    print(header)
    print('-'*100)
    for r in results:
        sig = '***' if r['bootstrap_p']<0.001 else ('**' if r['bootstrap_p']<0.01 else ('*' if r['bootstrap_p']<0.05 else 'n.s.'))
        print(f'{r["comparison"]:<45} {r["metric"]:<18} {r["diff"]:>+7.4f} {r["bootstrap_p"]:>10.6f} {r["wilcoxon_p"]:>10.6f} {r["cohens_d"]:>+8.4f} {sig:>5}')

if all_test_results_80:
    print_summary_table(all_test_results_80, 'TABLE: 80-Token Statistical Tests (ROUGE-L, Penalized-RL, Dedup-RL, Rep4)')
if all_bert_results_80:
    print_summary_table(all_bert_results_80, 'TABLE: 80-Token BERTScore F1 Tests')

if all_test_results_200:
    print_summary_table(all_test_results_200, 'TABLE: 200-Token Statistical Tests (ROUGE-L, Penalized-RL, Dedup-RL, Rep4)')
if all_bert_results_200:
    print_summary_table(all_bert_results_200, 'TABLE: 200-Token BERTScore F1 Tests')

# Key findings summary
print('\n' + '='*100)
print('KEY FINDINGS SUMMARY')
print('='*100)
all_results = all_test_results_80 + all_bert_results_80 + all_test_results_200 + all_bert_results_200
sig_count = sum(1 for r in all_results if r['significant'])
total_count = len(all_results)
print(f'\nTotal tests run: {total_count}')
print(f'Statistically significant (p < 0.05): {sig_count}/{total_count} ({sig_count/total_count*100:.0f}%)')

# Focus on key comparisons
for r in all_results:
    if 'Early-Stop' in r['comparison'] and 'Full' in r['comparison']:
        sig = 'YES (p<0.05)' if r['significant'] else 'NO (n.s.)'
        print(f'  [{r["metric"]}] {r["comparison"]}: diff={r["diff"]:+.4f}, p={r["bootstrap_p"]:.6f} -> Significant? {sig}')

In [ ]:
# Cell 10: METHOD-LEVEL SUMMARY WITH CONFIDENCE INTERVALS
print('='*100)
print('METHOD-LEVEL SUMMARY WITH 95% CONFIDENCE INTERVALS')
print('='*100)

def method_summary_with_ci(data_dict, label):
    print(f'\n{label}')
    print('-'*100)
    header = f'{"Method":<25} {"ROUGE-L":<18} {"Pen-ROUGE-L":<18} {"Dedup-RL":<18} {"Rep4gram":<16} {"BERTScore":<18}'
    print(header)
    print('-'*100)
    for method, records in data_dict.items():
        def ci_str(values):
            m = np.mean(values)
            se = np.std(values, ddof=1) / np.sqrt(len(values))
            lo = m - 1.96*se
            hi = m + 1.96*se
            return f'{m:.2f} [{lo:.2f},{hi:.2f}]'
        
        rl = [r['rouge_l_recomputed'] for r in records]
        pen = [r['rouge_l_penalized'] for r in records]
        ded = [r['rouge_l_dedup'] for r in records]
        rep = [r['rep4gram'] for r in records]
        
        bs_str = 'N/A'
        if 'bertscore_f1' in records[0]:
            bs = [r['bertscore_f1'] for r in records]
            bs_str = ci_str(bs)
        
        def ci_str_short(values):
            m = np.mean(values)
            se = np.std(values, ddof=1) / np.sqrt(len(values))
            return f'{m:.4f}+/-{1.96*se:.4f}'
        
        print(f'{method:<25} {ci_str(rl):<18} {ci_str(pen):<18} {ci_str(ded):<18} {ci_str_short(rep):<16} {bs_str:<18}')

if data_80tok:
    method_summary_with_ci(data_80tok, '80-TOKEN METHODS (Phase 3B)')
if data_200tok:
    method_summary_with_ci(data_200tok, '200-TOKEN METHODS (Phase 3C)')

In [ ]:
# Cell 11: EXPORT ALL RESULTS
print('='*70)
print('EXPORTING RESULTS')
print('='*70)

export = {
    'experiment': 'Statistical Significance Tests for Activation Steering',
    'n_bootstrap': N_BOOTSTRAP,
    'confidence_level': 1 - ALPHA_CI,
    'tests_80tok': all_test_results_80,
    'tests_200tok': all_test_results_200,
    'bertscore_tests_80tok': all_bert_results_80,
    'bertscore_tests_200tok': all_bert_results_200,
}

# Convert tuples to lists for JSON serialization
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_serializable(i) for i in obj]
    elif isinstance(obj, tuple):
        return list(obj)
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    return obj

export = make_serializable(export)

with open(os.path.join(OUTPUT_DIR, 'statistical_significance_results.json'), 'w') as f:
    json.dump(export, f, indent=2)

print('Exported: statistical_significance_results.json')
print('\nDONE! All statistical tests completed successfully!')